# Sistema de Semáforo Inteligente con Detección de Objetos
## Proyecto Final - Diplomado en Tecnologías para Inteligencia Artificial
**Universidad del Bío-Bío**

---

### Descripción
Sistema de visión por computador para controlar un semáforo de intersección de forma adaptativa. Detecta tres clases de objetos en tiempo real:
1. **Peatones**
2. **Vehículos** (autos, camionetas, SUV, camiones, motos, etc.)
3. **Vehículos de emergencia con balizas encendidas** (ambulancia, bomberos, carabineros)

La lógica de priorización es:
1. Si detecta un vehículo de emergencia con balizas → semáforo en verde en su carril (máxima prioridad).
2. Si no, prioriza el carril con mayor concentración de vehículos.
3. Por último, asigna verde al carril peatonal cuando hay alta concentración de peatones.

### Estructura del notebook
Cada sección es ejecutable de forma independiente para facilitar la depuración y la presentación.

| Sección | Contenido |
|---|---|
| 0 | Setup e instalación de dependencias |
| 1 | Importación de librerías y configuración |
| 2 | Carga y exploración del dataset |
| 3 | Análisis exploratorio (EDA) y visualización |
| 4 | División de datos (train/val 70/30) |
| 5 | Tratamiento de datos y data augmentation (con condiciones climáticas) |
| 5B | Detector de baja visibilidad (niebla, lluvia, noche) |
| 6 | Configuración del modelo YOLOv8 |
| 7 | Entrenamiento (transfer learning) |
| 8 | Evaluación: matriz de confusión y métricas |
| 9 | Testeo sobre imágenes nuevas |
| 10 | Módulo de lógica de priorización con falla segura |
| 11 | Demo end-to-end con manejo de visibilidad |
| 12 | Conclusiones y próximos pasos |

### Robustez climática
Concepción (Chile) presenta neblina densa frecuente en mañanas invernales, junto con lluvia intensa y baja visibilidad nocturna. El sistema incorpora tres estrategias complementarias para operar de forma confiable en estas condiciones:

1. **Augmentation climático** durante el entrenamiento (niebla sintética, lluvia, blur, baja luminancia) usando Albumentations.
2. **Detector de baja visibilidad** que mide en cada frame si la imagen es confiable (varianza del Laplaciano, contraste, brillo).
3. **Falla segura**: si la visibilidad es baja, el sistema revierte automáticamente al ciclo de tiempo fijo en vez de tomar decisiones basadas en detecciones poco confiables.

---
## Sección 0 — Setup e instalación

Instala las dependencias necesarias. El notebook corre en **PC local** (con o sin GPU) o en Google Colab.

Si usas GPU NVIDIA asegúrate de tener CUDA instalado. Sin GPU el sistema corre en CPU (más lento pero funcional para prototipo).

In [ ]:
# Instalación de dependencias principales
# Ultralytics ya trae torch, torchvision, opencv, matplotlib, etc.
%pip install -q ultralytics==8.3.0
%pip install -q roboflow
%pip install -q seaborn pandas scikit-learn
%pip install -q albumentations==1.4.18

print("✅ Dependencias instaladas")

---
## Sección 1 — Importación de librerías y configuración

Importamos las librerías base. Definimos semilla aleatoria para reproducibilidad.

In [ ]:
import os, random, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import torch
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
import albumentations as A

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Dispositivo (CPU o GPU automático)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE} | PyTorch: {torch.__version__}")

# Clases del proyecto
CLASES         = {0: "peaton", 1: "vehiculo", 2: "vehiculo_emergencia"}
COLORES_CLASES = {0: (0,255,255), 1: (255,0,0), 2: (0,0,255)}

print("\n📋 Clases del modelo:")
for k, v in CLASES.items():
    print(f"   {k}: {v}")

---
## Sección 2 — Carga y exploración del dataset

### Datasets seleccionados (todos CC BY 4.0 — totalmente libres)

Esta sección usa únicamente datasets con licencia **Creative Commons Attribution 4.0** (CC BY 4.0), que permiten descarga gratuita, uso académico y comercial con atribución. Detalles completos en `DATASETS.md`.

| Dataset | Aporta | Licencia | URL |
|---|---|---|---|
| **Open Images V7** (Google) | Peatones + vehículos + algunas ambulancias (subset filtrado ~10K img) | CC BY 4.0 / CC BY 2.0 | [storage.googleapis.com/openimages](https://storage.googleapis.com/openimages/web/index.html) |
| **Emergency Vehicles xockh** | Vehículos de emergencia con balizas on/off (798 img) | CC BY 4.0 ✅ | [Roboflow](https://universe.roboflow.com/yolo-fn1iu/emergency-vehicles-detection-xockh-af7sr) |

Total combinado: ~6.000–11.000 imágenes, suficiente para entrenar YOLOv8s con transfer learning.

**Bonus:** YOLOv8 ya viene preentrenado en **COCO** (CC BY 4.0), así que al cargar `yolov8s.pt` y hacer transfer learning ya estamos aprovechando ese dataset indirectamente.

### Para robustez climática (sin datasets no-comerciales)

Como **Foggy Cityscapes** y **BDD100K** son no-comerciales, en su lugar usamos:
1. **Augmentation sintético con Albumentations** (Sección 5 más abajo): `RandomFog`, `RandomRain`, etc.
2. **Filtrar Open Images V7** por escenas con condiciones climáticas adversas usando FiftyOne.

### Estructura YOLO esperada

```
dataset/
├── images/
│   ├── train/  *.jpg
│   └── val/    *.jpg
├── labels/
│   ├── train/  *.txt
│   └── val/    *.txt
└── data.yaml
```

**Antes de descargar:** crea cuenta gratis en [Roboflow](https://roboflow.com) y copia tu API key desde [Settings → API](https://app.roboflow.com/settings/api). Para Open Images V7 con FiftyOne no se necesita cuenta.

In [ ]:
# Dataset 1: Open Images V7 — ya descargado (3000 train + 500 val imágenes)
import os
OIV7_PATH = './datasets/openimages_yolo'
n_train = len([f for f in os.listdir(f'{OIV7_PATH}/train/images/val') if f.endswith(('.jpg','.png'))]) if os.path.exists(f'{OIV7_PATH}/train/images/val') else 0
n_val   = len([f for f in os.listdir(f'{OIV7_PATH}/val/images/val')   if f.endswith(('.jpg','.png'))]) if os.path.exists(f'{OIV7_PATH}/val/images/val') else 0
print(f'✅ Open Images V7: {n_train} train | {n_val} val → {OIV7_PATH}')

In [ ]:
# Dataset 2: Emergency Vehicles xockh — ya descargado (798 imágenes, CC BY 4.0)
EMERGENCIA_PATH = './datasets/emergency_vehicles/Emergency-Vehicles-Detection--1'
n_ev = len([f for f in os.listdir(f'{EMERGENCIA_PATH}/train/images') if f.endswith(('.jpg','.png'))]) if os.path.exists(f'{EMERGENCIA_PATH}/train/images') else 0
print(f'✅ Emergency Vehicles: {n_ev} imágenes train → {EMERGENCIA_PATH}')

In [ ]:
# ✅ Datasets ya combinados en dataset_combinado/
# Remapeo aplicado:
#   Open Images V7:       Person→0, Car/Truck/Bus/Motorcycle/Van→1, Ambulance→2
#   Emergency Vehicles:   ambulance_off/firetruck_off→1, ambulance_on/firetruck_on→2

import os
n_tr = len([f for f in os.listdir('./dataset_combinado/images/train') if f.endswith(('.jpg','.png'))]) if os.path.exists('./dataset_combinado/images/train') else 0
n_va = len([f for f in os.listdir('./dataset_combinado/images/val')   if f.endswith(('.jpg','.png'))]) if os.path.exists('./dataset_combinado/images/val')   else 0
print(f'✅ dataset_combinado listo: {n_tr} train | {n_va} val')

DATASET_PATH = "./dataset_combinado"
print(f"DATASET_PATH: {os.path.abspath(DATASET_PATH)}")

In [ ]:
# Conteo rápido de imágenes y etiquetas
def contar_archivos(carpeta, extensiones):
    if not os.path.exists(carpeta):
        return 0
    return sum(1 for f in os.listdir(carpeta) if f.lower().endswith(extensiones))

img_train = contar_archivos(f'{DATASET_PATH}/images/train', ('.jpg', '.jpeg', '.png'))
img_val = contar_archivos(f'{DATASET_PATH}/images/val', ('.jpg', '.jpeg', '.png'))
lbl_train = contar_archivos(f'{DATASET_PATH}/labels/train', ('.txt',))
lbl_val = contar_archivos(f'{DATASET_PATH}/labels/val', ('.txt',))

print("=" * 50)
print(f"📊 Resumen del dataset")
print("=" * 50)
print(f"Imágenes entrenamiento: {img_train}")
print(f"Etiquetas entrenamiento: {lbl_train}")
print(f"Imágenes validación:    {img_val}")
print(f"Etiquetas validación:   {lbl_val}")
print(f"Total imágenes:         {img_train + img_val}")
print("=" * 50)

---
## Sección 3 — Análisis exploratorio (EDA) y visualización

Aplicamos el enfoque de EDA**:
- Distribución de clases (importante para detectar desbalance)
- Distribución de tamaños de bounding boxes
- Visualización de muestras representativas

El desbalance de clases es crítico aquí: los vehículos de emergencia con balizas son raros respecto a peatones y vehículos comunes.

In [ ]:
def parsear_etiquetas(carpeta_labels):
    """Lee todos los .txt y devuelve un DataFrame con clase, bbox y archivo."""
    registros = []
    if not os.path.exists(carpeta_labels):
        return pd.DataFrame(registros)
    for archivo in os.listdir(carpeta_labels):
        if not archivo.endswith('.txt'):
            continue
        ruta = os.path.join(carpeta_labels, archivo)
        with open(ruta) as f:
            for linea in f:
                partes = linea.strip().split()
                if len(partes) >= 5:
                    registros.append({
                        'archivo': archivo,
                        'clase_id': int(float(partes[0])),
                        'x_center': float(partes[1]),
                        'y_center': float(partes[2]),
                        'width': float(partes[3]),
                        'height': float(partes[4]),
                        'area': float(partes[3]) * float(partes[4])
                    })
    df = pd.DataFrame(registros)
    if not df.empty:
        df['clase'] = df['clase_id'].map(CLASES)
    return df

df_train = parsear_etiquetas(f'{DATASET_PATH}/labels/train')
df_val = parsear_etiquetas(f'{DATASET_PATH}/labels/val')

print(f"Anotaciones train: {len(df_train)}")
print(f"Anotaciones val:   {len(df_val)}")
df_train.head()

In [ ]:
# Distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if not df_train.empty:
    sns.countplot(data=df_train, x='clase', ax=axes[0],
                  order=['peaton', 'vehiculo', 'vehiculo_emergencia'])
    axes[0].set_title('Distribución de clases — Entrenamiento')
    axes[0].set_xlabel('Clase')
    axes[0].set_ylabel('Cantidad de anotaciones')
    for p in axes[0].patches:
        axes[0].annotate(int(p.get_height()),
                         (p.get_x() + p.get_width()/2, p.get_height()),
                         ha='center', va='bottom')

if not df_val.empty:
    sns.countplot(data=df_val, x='clase', ax=axes[1],
                  order=['peaton', 'vehiculo', 'vehiculo_emergencia'])
    axes[1].set_title('Distribución de clases — Validación')
    axes[1].set_xlabel('Clase')
    axes[1].set_ylabel('Cantidad de anotaciones')
    for p in axes[1].patches:
        axes[1].annotate(int(p.get_height()),
                         (p.get_x() + p.get_width()/2, p.get_height()),
                         ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Detección de desbalance
if not df_train.empty:
    proporciones = df_train['clase'].value_counts(normalize=True) * 100
    print("\nProporciones (%):")
    print(proporciones.round(2))
    if proporciones.min() < 10:
        print("\n⚠️  Hay clases con baja representación. Aplicar:")
        print("   • Data augmentation más agresivo en clase minoritaria")
        print("   • Pesos de clase (class weights) en el entrenamiento")
        print("   • Considerar oversampling de la clase 'vehiculo_emergencia'")

In [ ]:
# Distribución de tamaños de bounding boxes
if not df_train.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    sns.histplot(data=df_train, x='width', hue='clase', kde=True, ax=axes[0], bins=30)
    axes[0].set_title('Distribución de ancho de bbox (normalizado)')
    
    sns.histplot(data=df_train, x='height', hue='clase', kde=True, ax=axes[1], bins=30)
    axes[1].set_title('Distribución de alto de bbox (normalizado)')
    
    sns.scatterplot(data=df_train, x='width', y='height', hue='clase',
                    alpha=0.5, ax=axes[2])
    axes[2].set_title('Ancho vs Alto por clase')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualizar muestras del dataset con sus anotaciones
def dibujar_anotaciones(img_path, label_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for linea in f:
                partes = linea.strip().split()
                if len(partes) < 5:
                    continue
                clase_id = int(float(partes[0]))
                xc, yc, bw, bh = map(float, partes[1:5])
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                color = COLORES_CLASES.get(clase_id, (255, 255, 255))
                color_rgb = (color[2], color[1], color[0])
                cv2.rectangle(img, (x1, y1), (x2, y2), color_rgb, 2)
                cv2.putText(img, CLASES.get(clase_id, '?'),
                            (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, color_rgb, 2)
    return img

# Mostrar 6 muestras aleatorias
carpeta_img = f'{DATASET_PATH}/images/train'
if os.path.exists(carpeta_img):
    imgs = [f for f in os.listdir(carpeta_img) if f.lower().endswith(('.jpg', '.png'))]
    muestras = random.sample(imgs, min(6, len(imgs)))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, nombre in zip(axes.flatten(), muestras):
        img_path = os.path.join(carpeta_img, nombre)
        label_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        img = dibujar_anotaciones(img_path, label_path)
        ax.imshow(img)
        ax.set_title(nombre, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## Sección 4 — División de datos (train/val 70/30)

Si el dataset descargado no viene dividido, hacemos la división **estratificada** según las clases presentes en cada imagen, para preservar la proporción de cada clase en ambos conjuntos.

**Buena práctica:** además del 70/30 train/val, idealmente reservar un 10% adicional como **test set** que no se toque hasta la evaluación final.

In [ ]:
def dividir_dataset(carpeta_origen_imgs, carpeta_origen_labels,
                    destino, ratio_train=0.7, ratio_val=0.2, ratio_test=0.1):
    """Divide imágenes y labels en train/val/test conservando emparejamiento."""
    imgs = [f for f in os.listdir(carpeta_origen_imgs) 
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    # Asignar clase dominante por imagen para estratificar
    clase_por_imagen = []
    for img in imgs:
        lbl = os.path.join(carpeta_origen_labels, img.rsplit('.', 1)[0] + '.txt')
        clases = []
        if os.path.exists(lbl):
            with open(lbl) as f:
                for linea in f:
                    p = linea.strip().split()
                    if p:
                        clases.append(int(p[0]))
        clase_dom = Counter(clases).most_common(1)[0][0] if clases else -1
        clase_por_imagen.append(clase_dom)
    
    # Split estratificado train vs (val+test)
    X_train, X_resto, y_train, y_resto = train_test_split(
        imgs, clase_por_imagen,
        train_size=ratio_train,
        stratify=clase_por_imagen if len(set(clase_por_imagen)) > 1 else None,
        random_state=SEED
    )
    # Split val vs test
    ratio_val_en_resto = ratio_val / (ratio_val + ratio_test)
    X_val, X_test, _, _ = train_test_split(
        X_resto, y_resto,
        train_size=ratio_val_en_resto,
        stratify=y_resto if len(set(y_resto)) > 1 else None,
        random_state=SEED
    )
    
    # Copiar archivos
    splits = {'train': X_train, 'val': X_val, 'test': X_test}
    for split, archivos in splits.items():
        os.makedirs(f'{destino}/images/{split}', exist_ok=True)
        os.makedirs(f'{destino}/labels/{split}', exist_ok=True)
        for archivo in archivos:
            shutil.copy(os.path.join(carpeta_origen_imgs, archivo),
                        f'{destino}/images/{split}/{archivo}')
            lbl_src = os.path.join(carpeta_origen_labels, archivo.rsplit('.', 1)[0] + '.txt')
            if os.path.exists(lbl_src):
                shutil.copy(lbl_src, f'{destino}/labels/{split}/')
    
    print(f"✅ División completada:")
    print(f"   Train: {len(X_train)} ({len(X_train)/len(imgs)*100:.1f}%)")
    print(f"   Val:   {len(X_val)} ({len(X_val)/len(imgs)*100:.1f}%)")
    print(f"   Test:  {len(X_test)} ({len(X_test)/len(imgs)*100:.1f}%)")
    return splits

# Ejecutar si necesitas re-dividir. Si el dataset ya viene dividido, omitir.
# splits = dividir_dataset('/content/raw/images', '/content/raw/labels', DATASET_PATH)
print("Función lista. Ejecutar si necesitas dividir manualmente.")

---
## Sección 5 — Tratamiento de datos y data augmentation

YOLOv8 aplica automáticamente data augmentation durante el entrenamiento. Sus técnicas incluyen:
- **Mosaic** (combinar 4 imágenes en una): aumenta variedad de contextos
- **MixUp** (mezcla de dos imágenes): mejora generalización
- **HSV** (variación de tono/saturación/valor): robustez a cambios de iluminación
- **Flip horizontal**: invarianza espacial
- **Mosaic + scale + translate**: variación de tamaño y posición

Estas técnicas son fundamentales aquí porque:
- Las balizas de emergencia generan reflejos y variaciones de iluminación que el HSV augmentation simula.
- Los vehículos aparecen en múltiples ángulos y distancias (mosaic ayuda).

### Augmentation climático para Concepción

Como el sistema se desplegará en una zona con **neblina densa frecuente** (mañanas invernales en Concepción), **lluvia intensa** y **noche**, las augmentations por defecto de YOLOv8 no son suficientes. Agregamos un pipeline de **Albumentations** que aplica de forma probabilística:
- `RandomFog`: simula niebla con densidad variable
- `RandomRain`: gotas y trazos de lluvia
- `RandomShadow`: sombras dinámicas
- `RandomBrightnessContrast`: simula amaneceres, atardeceres y noches
- `MotionBlur` y `GaussianBlur`: vibración de cámara y desenfoque por neblina densa
- `ISONoise`: ruido típico de cámaras a baja luz

Estas transformaciones se aplican **offline** sobre el dataset (generando copias aumentadas) para que YOLOv8 las consuma como imágenes normales durante el entrenamiento.

Definimos primero el archivo `data.yaml` requerido por YOLO:

In [ ]:
# Crear archivo de configuración data.yaml
data_yaml = f"""# Configuración del dataset para YOLOv8
path: {os.path.abspath(DATASET_PATH)}
train: images/train
val: images/val
test: images/test  # opcional

# Número de clases
nc: 3

# Nombres de las clases
names:
  0: peaton
  1: vehiculo
  2: vehiculo_emergencia
"""

yaml_path = f'{DATASET_PATH}/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(data_yaml)

print(f"✅ Archivo creado: {yaml_path}")
print("\nContenido:")
print(data_yaml)

In [ ]:
# Visualizar cómo se ve un mosaic batch antes de entrenar
# (esto se hace automáticamente, pero podemos visualizar manualmente)

def mostrar_mosaic_simulado(carpeta_img, n=4):
    """Simula visualmente una augmentación tipo mosaic combinando 4 imágenes."""
    imgs_paths = [os.path.join(carpeta_img, f) 
                  for f in os.listdir(carpeta_img) 
                  if f.lower().endswith(('.jpg', '.png'))][:n]
    if len(imgs_paths) < 4:
        print("No hay suficientes imágenes")
        return
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    for ax, p in zip(axes.flatten(), imgs_paths[:4]):
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle('Simulación: 4 imágenes combinadas en un mosaic batch')
    plt.tight_layout()
    plt.show()

if os.path.exists(f'{DATASET_PATH}/images/train'):
    mostrar_mosaic_simulado(f'{DATASET_PATH}/images/train')

In [ ]:
# ============================================================
# Pipeline de Albumentations para condiciones climáticas adversas
# ============================================================
# Estas transformaciones simulan niebla, lluvia, noche, blur — 
# condiciones reales en Concepción y otras zonas con clima variable.

# Probabilidades configurables: 30% de probabilidad por tipo de condición
# para que el modelo aprenda a generalizar sin perder la mayoría limpia.

pipeline_clima = A.Compose([
    # Condiciones de niebla — críticas en Concepción
    A.OneOf([
        A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.4, alpha_coef=0.08, p=1.0),  # niebla leve
        A.RandomFog(fog_coef_lower=0.4, fog_coef_upper=0.7, alpha_coef=0.10, p=1.0),  # niebla densa
    ], p=0.30),
    
    # Lluvia
    A.OneOf([
        A.RandomRain(slant_lower=-10, slant_upper=10, drop_length=15,
                     drop_width=1, drop_color=(200, 200, 200), blur_value=3, p=1.0),
        A.RandomRain(slant_lower=-20, slant_upper=20, drop_length=30,
                     drop_width=2, drop_color=(180, 180, 180), blur_value=5, p=1.0),  # lluvia fuerte
    ], p=0.20),
    
    # Cambios de luminancia: amanecer/atardecer/noche
    A.RandomBrightnessContrast(brightness_limit=(-0.4, 0.2), contrast_limit=(-0.3, 0.2), p=0.35),
    
    # Sombras dinámicas (árboles, postes, edificios)
    A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), num_shadows_lower=1, num_shadows_upper=3, p=0.20),
    
    # Blur por vibración / niebla densa
    A.OneOf([
        A.MotionBlur(blur_limit=7, p=1.0),
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
    ], p=0.20),
    
    # Ruido típico de cámaras a baja luz
    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.15),
    
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

print('✅ Pipeline de augmentation climático configurado')
print(f'   Transformaciones: {len(pipeline_clima.transforms)} bloques')

In [ ]:
# Visualizar el efecto del augmentation climático sobre una imagen
def visualizar_augmentation_clima(img_path, label_path, n_muestras=5):
    """Genera n versiones aumentadas de la misma imagen para comparar."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Leer bboxes
    bboxes = []
    class_labels = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for linea in f:
                p = linea.strip().split()
                if len(p) >= 5:
                    bboxes.append([float(p[1]), float(p[2]), float(p[3]), float(p[4])])
                    class_labels.append(int(float(p[0])))
    
    fig, axes = plt.subplots(1, n_muestras + 1, figsize=(4 * (n_muestras + 1), 4))
    axes[0].imshow(img)
    axes[0].set_title('Original', fontweight='bold')
    axes[0].axis('off')
    
    for i in range(n_muestras):
        try:
            aug = pipeline_clima(image=img, bboxes=bboxes, class_labels=class_labels)
            axes[i + 1].imshow(aug['image'])
            axes[i + 1].set_title(f'Aug {i + 1}')
            axes[i + 1].axis('off')
        except Exception as e:
            axes[i + 1].set_title(f'Error: {e}')
            axes[i + 1].axis('off')
    plt.tight_layout()
    plt.show()

# Aplicar sobre una imagen del dataset
carpeta_img = f'{DATASET_PATH}/images/train'
if os.path.exists(carpeta_img):
    imgs = [f for f in os.listdir(carpeta_img) if f.lower().endswith(('.jpg', '.png'))]
    if imgs:
        ej_img = os.path.join(carpeta_img, random.choice(imgs))
        ej_lbl = ej_img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        visualizar_augmentation_clima(ej_img, ej_lbl, n_muestras=5)

In [ ]:
# ============================================================
# Aplicar augmentation climático OFFLINE: genera copias adicionales
# del set de entrenamiento con condiciones adversas.
# ============================================================
# Estas copias se guardan como imágenes/labels nuevos en la misma carpeta
# de entrenamiento, así YOLOv8 las consume automáticamente.

def aplicar_augmentation_offline(carpeta_imgs, carpeta_labels, n_copias_por_img=2, prefijo='clima'):
    """Genera n_copias_por_img versiones aumentadas de cada imagen."""
    imgs = [f for f in os.listdir(carpeta_imgs) 
            if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith(prefijo)]
    
    generadas = 0
    for nombre in imgs:
        img_path = os.path.join(carpeta_imgs, nombre)
        lbl_path = os.path.join(carpeta_labels, nombre.rsplit('.', 1)[0] + '.txt')
        
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Leer anotaciones
        bboxes, class_labels = [], []
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for linea in f:
                    p = linea.strip().split()
                    if len(p) >= 5:
                        bboxes.append([float(p[1]), float(p[2]), float(p[3]), float(p[4])])
                        class_labels.append(int(p[0]))
        
        for i in range(n_copias_por_img):
            try:
                aug = pipeline_clima(image=img, bboxes=bboxes, class_labels=class_labels)
                nuevo_nombre = f'{prefijo}_{i}_{nombre}'
                cv2.imwrite(os.path.join(carpeta_imgs, nuevo_nombre),
                            cv2.cvtColor(aug['image'], cv2.COLOR_RGB2BGR))
                # Guardar label nuevo
                lbl_nuevo = os.path.join(carpeta_labels, f'{prefijo}_{i}_' + nombre.rsplit('.', 1)[0] + '.txt')
                with open(lbl_nuevo, 'w') as f:
                    for bbox, cls in zip(aug['bboxes'], aug['class_labels']):
                        f.write(f'{cls} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n')
                generadas += 1
            except Exception:
                continue
    
    print(f'✅ Generadas {generadas} imágenes con augmentation climático')
    return generadas

aplicar_augmentation_offline(
    f'{DATASET_PATH}/images/train',
    f'{DATASET_PATH}/labels/train',
    n_copias_por_img=2
)

---
## Sección 5B — Detector de baja visibilidad

Aunque entrenemos con augmentation climático, el modelo puede fallar bajo niebla extrema o ausencia total de luz. Un buen sistema de control de tráfico debe **saber cuándo no confiar en sus propias detecciones**.

Implementamos un detector liviano basado en tres métricas clásicas de calidad de imagen:

| Métrica | Mide | Umbral típico |
|---|---|---|
| **Varianza del Laplaciano** | Nitidez / borrosidad | < 50 → imagen borrosa |
| **Contraste RMS** | Diferencia entre claros y oscuros | < 25 → bajo contraste (niebla) |
| **Brillo medio** | Luminancia global | < 40 → muy oscuro, > 215 → sobreexpuesto |

Estas métricas son **rápidas** (< 1 ms por frame) y se calculan sobre la imagen completa antes de pasarla a YOLOv8. Si la imagen no supera el umbral mínimo de confiabilidad, el sistema entra en modo de **falla segura** y revierte al ciclo de tiempo fijo del semáforo.

> **Principio de diseño:** un semáforo debe operar de forma predecible incluso cuando el modelo de IA no es confiable. Nunca debe quedar bloqueado ni tomar decisiones arriesgadas.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class CondicionVisibilidad(Enum):
    NORMAL = 'normal'
    BAJA = 'baja_visibilidad'
    CRITICA = 'visibilidad_critica'

@dataclass
class ConfigVisibilidad:
    # Umbrales calibrados — ajustar según pruebas en terreno
    umbral_nitidez_baja: float = 100.0      # Laplaciano < esto → baja nitidez
    umbral_nitidez_critica: float = 50.0    # Laplaciano < esto → muy borrosa
    umbral_contraste_bajo: float = 35.0     # RMS contrast < esto → bajo contraste
    umbral_contraste_critico: float = 20.0  # RMS contrast < esto → niebla densa probable
    umbral_brillo_min: float = 40.0         # Demasiado oscura
    umbral_brillo_max: float = 215.0        # Sobreexpuesta

class DetectorVisibilidad:
    """Estima la calidad de una imagen para decidir si confiar en la detección."""
    
    def __init__(self, config: ConfigVisibilidad = None):
        self.config = config or ConfigVisibilidad()
    
    def calcular_metricas(self, img_bgr):
        """Calcula nitidez, contraste y brillo de una imagen BGR."""
        gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        
        # Varianza del Laplaciano → nitidez
        nitidez = cv2.Laplacian(gris, cv2.CV_64F).var()
        
        # Contraste RMS
        contraste = float(gris.std())
        
        # Brillo medio
        brillo = float(gris.mean())
        
        return {'nitidez': nitidez, 'contraste': contraste, 'brillo': brillo}
    
    def evaluar(self, img_bgr):
        """Clasifica la imagen en NORMAL / BAJA / CRITICA y entrega diagnóstico."""
        m = self.calcular_metricas(img_bgr)
        c = self.config
        
        razones = []
        nivel = CondicionVisibilidad.NORMAL
        
        # Casos críticos (cualquiera dispara)
        if m['nitidez'] < c.umbral_nitidez_critica:
            nivel = CondicionVisibilidad.CRITICA
            razones.append(f'Nitidez muy baja ({m["nitidez"]:.1f})')
        if m['contraste'] < c.umbral_contraste_critico:
            nivel = CondicionVisibilidad.CRITICA
            razones.append(f'Contraste crítico ({m["contraste"]:.1f}) — posible niebla densa')
        if m['brillo'] < c.umbral_brillo_min:
            nivel = CondicionVisibilidad.CRITICA
            razones.append(f'Imagen muy oscura ({m["brillo"]:.1f})')
        if m['brillo'] > c.umbral_brillo_max:
            nivel = CondicionVisibilidad.CRITICA
            razones.append(f'Imagen sobreexpuesta ({m["brillo"]:.1f})')
        
        # Si no es crítica, evaluar si es baja
        if nivel == CondicionVisibilidad.NORMAL:
            if m['nitidez'] < c.umbral_nitidez_baja:
                nivel = CondicionVisibilidad.BAJA
                razones.append(f'Nitidez reducida ({m["nitidez"]:.1f})')
            if m['contraste'] < c.umbral_contraste_bajo:
                nivel = CondicionVisibilidad.BAJA
                razones.append(f'Contraste reducido ({m["contraste"]:.1f}) — posible neblina')
        
        return {
            'nivel': nivel,
            'metricas': m,
            'razones': razones if razones else ['Condiciones normales'],
            'confiable': nivel != CondicionVisibilidad.CRITICA
        }

# Probar el detector sobre imágenes con distintos niveles de aug climático
detector_vis = DetectorVisibilidad()

# Caso 1: imagen original
if os.path.exists(carpeta_img):
    imgs = [f for f in os.listdir(carpeta_img) if f.lower().endswith(('.jpg', '.png'))]
    if imgs:
        ej_img_path = os.path.join(carpeta_img, random.choice(imgs))
        img_orig = cv2.imread(ej_img_path)
        
        # Simular distintos niveles de niebla manualmente
        casos = {'Original': img_orig}
        
        # Niebla leve
        aug_leve = A.RandomFog(fog_coef_lower=0.2, fog_coef_upper=0.3, p=1.0)
        casos['Niebla leve'] = cv2.cvtColor(
            aug_leve(image=cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))['image'],
            cv2.COLOR_RGB2BGR
        )
        
        # Niebla densa
        aug_densa = A.RandomFog(fog_coef_lower=0.7, fog_coef_upper=0.9, p=1.0)
        casos['Niebla densa'] = cv2.cvtColor(
            aug_densa(image=cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))['image'],
            cv2.COLOR_RGB2BGR
        )
        
        # Muy oscura
        aug_noche = A.RandomBrightnessContrast(brightness_limit=(-0.7, -0.6),
                                                contrast_limit=(-0.3, -0.2), p=1.0)
        casos['Noche extrema'] = cv2.cvtColor(
            aug_noche(image=cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))['image'],
            cv2.COLOR_RGB2BGR
        )
        
        # Mostrar resultados
        fig, axes = plt.subplots(1, len(casos), figsize=(5 * len(casos), 5))
        for ax, (nombre, img) in zip(axes, casos.items()):
            resultado = detector_vis.evaluar(img)
            color = {'normal': 'green', 'baja_visibilidad': 'orange',
                     'visibilidad_critica': 'red'}[resultado['nivel'].value]
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            titulo = f"{nombre}\n[{resultado['nivel'].value.upper()}]\n"
            titulo += f"N:{resultado['metricas']['nitidez']:.0f} "
            titulo += f"C:{resultado['metricas']['contraste']:.0f} "
            titulo += f"B:{resultado['metricas']['brillo']:.0f}"
            ax.set_title(titulo, color=color, fontweight='bold', fontsize=10)
            ax.axis('off')
        plt.tight_layout()
        plt.show()
        
        # Imprimir diagnóstico detallado
        for nombre, img in casos.items():
            r = detector_vis.evaluar(img)
            print(f'\n📊 {nombre}: {r["nivel"].value}')
            for razon in r['razones']:
                print(f'   • {razon}')
            print(f'   → ¿Confiar en detecciones? {"SÍ" if r["confiable"] else "NO (falla segura)"}')

---
## Sección 6 — Configuración del modelo YOLOv8

### Justificación de la elección del modelo

**YOLOv8** (You Only Look Once, versión 8 de Ultralytics) se eligió porque:

1. **One-stage detector**: hace detección y clasificación en una sola pasada → permite **inferencia en tiempo real** (>30 FPS en GPU), requisito indispensable para un semáforo.
2. **Estado del arte** en detección de objetos sobre datasets de tráfico (COCO, VisDrone).
3. **Transfer learning** disponible: pesos preentrenados en COCO (80 clases incluyendo persona y vehículos).
4. **API simple** y ampliamente usada en sistemas de transporte inteligente.
5. Compatible con despliegue en edge (Jetson Nano, Coral)

### Variantes disponibles

| Modelo | Parámetros | mAP COCO | Velocidad | Uso recomendado |
|---|---|---|---|---|
| YOLOv8n (nano) | 3.2M | 37.3 | ⚡⚡⚡ | Edge / prototipo |
| YOLOv8s (small) | 11.2M | 44.9 | ⚡⚡ | **Balance recomendado** |
| YOLOv8m (medium) | 25.9M | 50.2 | ⚡ | Mayor precisión |
| YOLOv8l (large) | 43.7M | 52.9 | 🐢 | Solo si hay GPU robusta |

Comenzamos con **YOLOv8s** como balance entre velocidad y precisión.

In [ ]:
# Cargar modelo preentrenado YOLOv8s (transfer learning desde COCO)
modelo = YOLO('yolov8s.pt')

print("✅ Modelo YOLOv8s cargado con pesos preentrenados en COCO")
print(f"   Arquitectura: {modelo.model.__class__.__name__}")
print(f"   Parámetros: {sum(p.numel() for p in modelo.model.parameters()):,}")

# Información del modelo
modelo.info()

---
## Sección 7 — Entrenamiento (transfer learning)

Aplicamos **transferencia de aprendizaje**: partimos de los pesos del modelo preentrenado en COCO y lo ajustamos a nuestras 3 clases.

### Hiperparámetros principales

- `epochs=50`: ajustar según convergencia (curvas de pérdida).
- `imgsz=640`: resolución estándar (640×640), balance velocidad/precisión.
- `batch=16`: ajustar según VRAM disponible.
- `patience=10`: **early stopping** (parar si no mejora en 10 épocas)
- `optimizer='AdamW'`: adaptativo, suele converger más rápido para fine-tuning.
- `lr0=0.01`: tasa de aprendizaje inicial (YOLOv8 default).
- `cos_lr=True`: scheduler coseno (decae suavemente).

In [ ]:
import os
from ultralytics import YOLO

yaml_path = os.path.abspath('./dataset_combinado/data.yaml')
SEED = 42

last_weights = 'runs/semaforo/exp2_yolov8s_50ep/weights/last.pt'
modelo_ft = YOLO(last_weights)

resultados = modelo_ft.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=4,
    patience=10,
    optimizer='AdamW',
    lr0=0.01,
    cos_lr=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    project='runs/semaforo',
    name='exp2_yolov8s_50ep',
    exist_ok=True,
    save=True,
    save_period=10,
    seed=SEED,
    deterministic=True,
    device=DEVICE,
    workers=0,
    verbose=True,
    resume=True,
)

print(f'\n✅ Entrenamiento completado: {resultados.save_dir}')


In [ ]:
# Visualizar las curvas de entrenamiento (loss, mAP, precision, recall)
from IPython.display import display as ipy_display, Image as IPImage
ipy_display(IPImage('runs/semaforo/exp2_yolov8s_50ep/results.png', width=900))


---
## Sección 8 — Evaluación: matriz de confusión y métricas

Métricas que vamos a analizar:

- **Precisión (Precision)**: de las detecciones positivas, cuántas son correctas.
- **Sensibilidad (Recall)**: de los objetos reales, cuántos detecta.
- **F1-score**: media armónica de precisión y recall.
- **mAP@0.5**: mean Average Precision con IoU≥0.5 (estándar de detección).
- **mAP@0.5:0.95**: promedio sobre múltiples umbrales (más estricto).
- **Matriz de confusión**: errores por clase, fundamental para evaluar la clase crítica `vehiculo_emergencia`.

> Para la aplicación, la métrica **más importante es el recall de `vehiculo_emergencia`** — un falso negativo (no detectar una ambulancia) tiene consecuencias graves.

In [ ]:
# Validar el modelo entrenado sobre el conjunto de validación
results_dir = 'runs/semaforo/exp2_yolov8s_50ep'
best_weights = os.path.join(results_dir, 'weights', 'best.pt')
if not os.path.exists(best_weights):
    best_weights = 'runs/detect/train/weights/best.pt'

print(f"Cargando pesos: {best_weights}")
modelo_eval = YOLO(best_weights)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
metricas = modelo_eval.val(data=yaml_path, split='val', device=DEVICE)
print("\n📊 Métricas globales sobre validación:")
print(f"   mAP@0.5      = {metricas.box.map50:.4f}")
print(f"   mAP@0.5:0.95 = {metricas.box.map:.4f}")
print(f"   Precisión    = {metricas.box.mp:.4f}")
print(f"   Recall       = {metricas.box.mr:.4f}")

In [ ]:
# Métricas por clase — clave para evaluar la clase crítica
print("\n📊 Métricas por clase:")
print(f"{'Clase':<25}{'P':>10}{'R':>10}{'mAP50':>10}{'mAP50-95':>12}")
print("-" * 67)
for i, nombre in CLASES.items():
    p = metricas.box.p[i] if i < len(metricas.box.p) else 0
    r = metricas.box.r[i] if i < len(metricas.box.r) else 0
    ap50 = metricas.box.ap50[i] if i < len(metricas.box.ap50) else 0
    ap = metricas.box.ap[i] if i < len(metricas.box.ap) else 0
    print(f"{nombre:<25}{p:>10.4f}{r:>10.4f}{ap50:>10.4f}{ap:>12.4f}")

In [ ]:
# Visualizar matriz de confusión generada automáticamente por YOLOv8
from IPython.display import Image as IPImage

cm_normalized = 'runs/semaforo/exp2_yolov8s_50ep/confusion_matrix_normalized.png'
cm_raw = 'runs/semaforo/exp2_yolov8s_50ep/confusion_matrix.png'

for cm_path in [cm_raw, cm_normalized]:
    if os.path.exists(cm_path):
        display(IPImage(cm_path))
    else:
        print(f"No encontrado: {cm_path}")

---
## Sección 9 — Testeo sobre imágenes nuevas

Probamos el modelo con imágenes que **no vio durante entrenamiento**. Esto valida la capacidad de generalización.

In [ ]:
from IPython.display import display as ipy_display

def predecir_y_mostrar(modelo, imagen_path, conf=0.25):
    """Ejecuta inferencia y muestra resultados visuales."""
    resultados = modelo.predict(imagen_path, conf=conf, device=DEVICE, verbose=False)
    r = resultados[0]

    img_anotada = r.plot()
    img_anotada = cv2.cvtColor(img_anotada, cv2.COLOR_BGR2RGB)

    # Redibujar cajas de emergencia en rojo
    if r.boxes is not None and len(r.boxes) > 0:
        for box in r.boxes:
            if int(box.cls[0]) == 2:
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                cv2.rectangle(img_anotada, (x1, y1), (x2, y2), (255, 0, 0), 3)

    conteo = Counter()
    if r.boxes is not None and len(r.boxes) > 0:
        for cls in r.boxes.cls.cpu().numpy().astype(int):
            conteo[CLASES[cls]] += 1

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img_anotada)
    titulo = f"Detecciones: {dict(conteo)}" if conteo else "Sin detecciones"
    ax.set_title(titulo)
    ax.axis('off')
    plt.tight_layout()
    ipy_display(fig)
    plt.close(fig)

    return r, conteo

# ─────────────────────────────────────────────────────────────
# Probar con imágenes del conjunto de val/
# ─────────────────────────────────────────────────────────────
carpeta_test = f'{DATASET_PATH}/images/test'
imgs_test_check = [f for f in os.listdir(carpeta_test)
                   if f.lower().endswith(('.jpg', '.png'))] if os.path.exists(carpeta_test) else []
if not imgs_test_check:
    carpeta_test = f'{DATASET_PATH}/images/val'

imgs_test = [f for f in os.listdir(carpeta_test) if f.lower().endswith(('.jpg', '.png'))]
print(f'Carpeta: {carpeta_test}')
print(f'Imágenes encontradas: {len(imgs_test)}')

for nombre in random.sample(imgs_test, min(3, len(imgs_test))):
    path = os.path.join(carpeta_test, nombre)
    print(f'\n🔍 Imagen: {nombre}')
    r, conteo = predecir_y_mostrar(modelo_eval, path, conf=0.25)


---
## Sección 10 — Módulo de lógica de priorización con falla segura

Esta es la **capa de negocio** del sistema: una vez que el modelo detecta los objetos y se ha evaluado la visibilidad, decidimos qué luz del semáforo poner en verde.

### Reglas (en orden de prioridad)

0. **Visibilidad crítica detectada (niebla densa, oscuridad extrema)** → **FALLA SEGURA**: revertir al ciclo de tiempo fijo del semáforo, ignorando las detecciones del modelo.
1. **Emergencia con balizas detectada** → verde en su carril, sin importar el conteo de otros.
2. **Densidad alta de vehículos** (umbral configurable) → verde para vehículos.
3. **Densidad alta de peatones** → verde para peatones.
4. **Estado por defecto** → ciclo normal del semáforo.

Se incluye una **ventana de estabilidad temporal**: la decisión solo se aplica si se sostiene durante N frames consecutivos, para evitar oscilaciones por detecciones espurias.

**Principio de diseño:** ante cualquier duda (visibilidad pobre o modelo poco confiable), el sistema **siempre revierte al comportamiento conservador**. Un semáforo que falla mal puede causar accidentes; un semáforo que se vuelve “tonto pero predecible” no.

In [ ]:
from collections import deque
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ConfigSemaforo:
    umbral_vehiculos: int = 5
    umbral_peatones: int = 3
    conf_minima_emergencia: float = 0.5
    conf_minima_general: float = 0.35
    ventana_frames: int = 5
    tiempo_min_verde_seg: float = 10.0

@dataclass
class EstadoSemaforo:
    decision_actual: str = 'CICLO_NORMAL'
    razon: str = 'Inicialización'
    historial_decisiones: deque = field(default_factory=lambda: deque(maxlen=5))
    timestamp_ultimo_cambio: float = 0.0
    
class ControladorSemaforo:
    def __init__(self, config: ConfigSemaforo = None,
                 detector_visibilidad: Optional['DetectorVisibilidad'] = None):
        self.config = config or ConfigSemaforo()
        self.detector_vis = detector_visibilidad
        self.estado = EstadoSemaforo()
        self.estado.historial_decisiones = deque(maxlen=self.config.ventana_frames)
    
    def evaluar(self, detecciones, img_bgr=None):
        """
        detecciones: lista de tuplas (clase_id, confianza)
        img_bgr: imagen BGR opcional para evaluar visibilidad
        Devuelve: decisión + razón + diagnóstico visibilidad.
        """
        # REGLA 0: Falla segura por visibilidad crítica
        diagnostico_vis = None
        if self.detector_vis is not None and img_bgr is not None:
            diagnostico_vis = self.detector_vis.evaluar(img_bgr)
            if not diagnostico_vis['confiable']:
                self.estado.decision_actual = 'FALLA_SEGURA'
                self.estado.razon = ('Visibilidad crítica — revirtiendo a ciclo fijo. '
                                     + '; '.join(diagnostico_vis['razones']))
                self.estado.historial_decisiones.append('FALLA_SEGURA')
                return {
                    'decision': self.estado.decision_actual,
                    'razon': self.estado.razon,
                    'conteos': {'peatones': 0, 'vehiculos': 0, 'emergencia': 0},
                    'visibilidad': diagnostico_vis
                }
        
        # Conteo de detecciones por clase
        n_peatones = sum(1 for c, conf in detecciones 
                         if c == 0 and conf >= self.config.conf_minima_general)
        n_vehiculos = sum(1 for c, conf in detecciones 
                          if c == 1 and conf >= self.config.conf_minima_general)
        n_emergencia = sum(1 for c, conf in detecciones 
                           if c == 2 and conf >= self.config.conf_minima_emergencia)
        
        # REGLA 1: emergencia tiene prioridad absoluta
        if n_emergencia > 0:
            decision = 'VERDE_EMERGENCIA'
            razon = f'Detectados {n_emergencia} vehículos de emergencia con balizas'
        # REGLA 2: alta densidad vehicular
        elif n_vehiculos >= self.config.umbral_vehiculos:
            decision = 'VERDE_VEHICULOS'
            razon = f'Alta densidad vehicular: {n_vehiculos} vehículos'
        # REGLA 3: alta densidad de peatones
        elif n_peatones >= self.config.umbral_peatones:
            decision = 'VERDE_PEATONES'
            razon = f'Alta densidad peatonal: {n_peatones} peatones'
        # REGLA 4: defecto
        else:
            decision = 'CICLO_NORMAL'
            razon = f'Tráfico normal (P:{n_peatones} V:{n_vehiculos} E:{n_emergencia})'
        
        # Si la visibilidad es baja (pero no crítica), aumentar umbrales y avisar
        if diagnostico_vis and diagnostico_vis['nivel'].value == 'baja_visibilidad':
            razon += ' [Atención: visibilidad reducida — decisiones más conservadoras]'
        
        # Ventana de estabilidad temporal
        self.estado.historial_decisiones.append(decision)
        
        # Emergencia: aplicar de inmediato
        if decision == 'VERDE_EMERGENCIA':
            self.estado.decision_actual = decision
            self.estado.razon = razon
        # Otras decisiones: requieren mayoría en la ventana
        elif len(self.estado.historial_decisiones) == self.config.ventana_frames:
            mas_comun, _ = Counter(self.estado.historial_decisiones).most_common(1)[0]
            if mas_comun == decision:
                self.estado.decision_actual = decision
                self.estado.razon = razon
        
        return {
            'decision': self.estado.decision_actual,
            'razon': self.estado.razon,
            'conteos': {'peatones': n_peatones, 'vehiculos': n_vehiculos, 'emergencia': n_emergencia},
            'visibilidad': diagnostico_vis
        }

# Demo rápida del controlador con falla segura
ctrl = ControladorSemaforo(detector_visibilidad=DetectorVisibilidad())

# Simular imagen normal y luego niebla densa
if os.path.exists(carpeta_img):
    imgs_demo = [f for f in os.listdir(carpeta_img) if f.lower().endswith(('.jpg', '.png'))]
    if imgs_demo:
        ej = cv2.imread(os.path.join(carpeta_img, imgs_demo[0]))
        ej_niebla = cv2.cvtColor(
            A.RandomFog(fog_coef_lower=0.8, fog_coef_upper=0.9, p=1.0)(
                image=cv2.cvtColor(ej, cv2.COLOR_BGR2RGB))['image'],
            cv2.COLOR_RGB2BGR
        )
        
        casos = [
            ('Imagen clara + ambulancia detectada', ej, [(1, 0.9), (2, 0.92)]),
            ('Niebla densa + ambulancia detectada (debería fallar seguro)', ej_niebla, [(1, 0.9), (2, 0.92)]),
            ('Imagen clara + atasco vehicular', ej, [(1, 0.9)] * 7),
        ]
        
        print('\n🚦 Demostración: el sistema sabe cuándo NO confiar en sus detecciones\n' + '=' * 70)
        for nombre, img, det in casos:
            ctrl = ControladorSemaforo(detector_visibilidad=DetectorVisibilidad())
            for _ in range(5):
                res = ctrl.evaluar(det, img)
            print(f'\n📌 {nombre}')
            print(f'   Decisión: {res["decision"]}')
            print(f'   Razón:    {res["razon"]}')
            if res.get('visibilidad'):
                print(f'   Visibilidad: {res["visibilidad"]["nivel"].value}')

---
## Sección 11 — Demo end-to-end (prototipo, 1 cámara)

> **Nota:** Esta sección demuestra el pipeline básico con una sola cámara como prototipo. Para la intersección completa con 2 cámaras ver la **Sección 12**.



Integramos detección + lógica de priorización + detector de visibilidad en una función que recibe una imagen y devuelve la decisión del semáforo, con visualización completa.

In [ ]:
def pipeline_semaforo(imagen_path, modelo, controlador, conf=0.25):
    """Pipeline completo: visibilidad → detección → decisión → visualización."""
    # Leer imagen original
    img_bgr = cv2.imread(imagen_path) if isinstance(imagen_path, str) else imagen_path
    
    # 1. Inferencia
    resultado = modelo.predict(img_bgr, conf=conf, device=DEVICE, verbose=False)[0]
    
    # 2. Extraer detecciones
    detecciones = []
    if resultado.boxes is not None and len(resultado.boxes) > 0:
        clases = resultado.boxes.cls.cpu().numpy().astype(int)
        confs = resultado.boxes.conf.cpu().numpy()
        detecciones = list(zip(clases, confs))
    
    # 3. Decisión del semáforo CON visibilidad (simular ventana llena)
    for _ in range(controlador.config.ventana_frames):
        decision = controlador.evaluar(detecciones, img_bgr)
    
    # 4. Visualización
    img_anotada = resultado.plot()
    img_anotada = cv2.cvtColor(img_anotada, cv2.COLOR_BGR2RGB)
    
    # Mapear decisión a color del semáforo
    colores_semaforo = {
        'VERDE_EMERGENCIA': ('🚨', 'red'),
        'VERDE_VEHICULOS':  ('🟢', 'green'),
        'VERDE_PEATONES':   ('🟢', 'blue'),
        'CICLO_NORMAL':     ('🟡', 'gray'),
        'FALLA_SEGURA':     ('⚠️', 'darkred')
    }
    emoji, color_titulo = colores_semaforo.get(decision['decision'], ('⚪', 'black'))
    
    # Construir título
    titulo = f"{emoji} {decision['decision']}\n{decision['razon']}"
    if decision.get('visibilidad'):
        m = decision['visibilidad']['metricas']
        titulo += f"\n[Visibilidad: {decision['visibilidad']['nivel'].value} | "
        titulo += f"N:{m['nitidez']:.0f} C:{m['contraste']:.0f} B:{m['brillo']:.0f}]"
    
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.imshow(img_anotada)
    ax.set_title(titulo, fontsize=12, color=color_titulo, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    from IPython.display import display as ipy_display
    ipy_display(fig)
    plt.close(fig)
    
    return decision


# Probar pipeline con varias imágenes — incluyendo casos de baja visibilidad simulados
if os.path.exists(carpeta_test):
    imgs_test = [f for f in os.listdir(carpeta_test) if f.lower().endswith(('.jpg', '.png'))]
    if imgs_test:
        muestra = random.choice(imgs_test)
        path_orig = os.path.join(carpeta_test, muestra)
        img_clara = cv2.imread(path_orig)
        img_niebla = cv2.cvtColor(
            A.RandomFog(fog_coef_lower=0.75, fog_coef_upper=0.9, p=1.0)(
                image=cv2.cvtColor(img_clara, cv2.COLOR_BGR2RGB))['image'],
            cv2.COLOR_RGB2BGR
        )
        
        print('=' * 70)
        print('CASO 1: Imagen clara → modelo debe operar normalmente')
        print('=' * 70)
        ctrl = ControladorSemaforo(detector_visibilidad=DetectorVisibilidad())
        pipeline_semaforo(img_clara, modelo_eval, ctrl)
        
        print('\n' + '=' * 70)
        print('CASO 2: Misma escena con niebla densa → debe activar FALLA SEGURA')
        print('=' * 70)
        ctrl = ControladorSemaforo(detector_visibilidad=DetectorVisibilidad())
        pipeline_semaforo(img_niebla, modelo_eval, ctrl)

In [ ]:
from IPython.display import Video as IPVideo, display as ipy_display

def procesar_video(video_path, modelo, salida='salida.mp4'):
    """Procesa un video frame por frame, guarda el resultado y lo muestra en el notebook."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"No se pudo abrir {video_path}")
        return

    fps      = int(cap.get(cv2.CAP_PROP_FPS))
    w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(salida, fourcc, fps, (w, h))

    ctrl      = ControladorSemaforo()
    historial = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        r = modelo.predict(frame, conf=0.25, device=DEVICE, verbose=False)[0]

        det = []
        if r.boxes is not None and len(r.boxes) > 0:
            cls = r.boxes.cls.cpu().numpy().astype(int)
            cf  = r.boxes.conf.cpu().numpy()
            det = list(zip(cls, cf))

        decision = ctrl.evaluar(det)
        historial.append(decision['decision'])

        frame_anot = r.plot()

        # Caja roja sobre vehiculo_emergencia
        if r.boxes is not None:
            for box in r.boxes:
                if int(box.cls[0]) == 2:
                    x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                    cv2.rectangle(frame_anot, (x1, y1), (x2, y2), (0, 0, 255), 3)

        cv2.putText(frame_anot, decision['decision'], (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        cv2.putText(frame_anot, decision['razon'], (20, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        out.write(frame_anot)

        frame_idx += 1
        if frame_idx % 30 == 0:
            print(f"   Procesado: {frame_idx}/{n_frames} frames")

    cap.release()
    out.release()
    print(f"\n✅ Video guardado en: {salida}")
    print(f"   Distribución de decisiones: {Counter(historial)}")
    # Re-encodear a H.264 para reproducir en el notebook
    import subprocess

    salida_h264 = salida.replace('.mp4', '_h264.mp4')
    subprocess.run([
        'ffmpeg', '-y', '-i', salida,
        '-vcodec', 'libx264', '-crf', '23',
        salida_h264
    ], capture_output=True)
    ipy_display(IPVideo(salida_h264, embed=True))


print("✅ procesar_video lista.")


In [ ]:
procesar_video(
    video_path='videos/Ambu1.mp4',
    modelo=modelo_eval,
    salida='videos/Ambu1_resultado2.mp4'
)

## Sección 12 — Control de intersección (2 cámaras)

El pipeline procesa **2 cámaras simultáneamente**, una por eje de tráfico. La intersección opera con:

- **cam_sur**: apunta al tráfico Sur-Norte.
- **cam_este**: apunta al tráfico Este-Oeste.
- **Peatones**: pueden cruzar en las 4 esquinas, gestionados implícitamente por fase.
- **Un controlador central** (`ControladorInterseccion`) que recibe las detecciones de ambas cámaras y decide la fase activa.

### Fases del semáforo

| Fase | Vehículos | Peatones | Trigger |
|---|---|---|---|
| **FASE_SN** | Verde N+S, Rojo E+O | Verde cruces E+O | Ciclo normal o densidad SN mayor |
| **FASE_EO** | Verde E+O, Rojo S+N | Verde cruces S+N | Ciclo normal o densidad EO mayor |
| **TRANSICION** | Amarillo ambos ejes | Rojo todas | Entre fases (3 s) |
| **EMERGENCIA** | Verde en eje de emergencia | Rojo todas | Vehículo emergencia detectado |
| **FALLA_SEGURA** | Amarillo parpadeante | Rojo todas | Cámara con visibilidad crítica |

### Reglas de priorización

0. **REGLA 0 (Falla segura):** si alguna cámara reporta visibilidad crítica → amarillo parpadeante.
1. **REGLA 1 (Emergencia):** si una cámara detecta vehículo de emergencia → fase Emergencia inmediata en ese eje.
2. **REGLA 2 (Densidad asimétrica):** si un eje tiene mucho más tráfico que el otro y se cumplió el tiempo mínimo verde → cambiar fase.
3. **REGLA 3 (Ciclo normal):** alternar FASE_SN y FASE_EO en duraciones nominales.

In [ ]:
import importlib, sys
if 'controlador_interseccion' in sys.modules:
    importlib.reload(sys.modules['controlador_interseccion'])
from controlador_interseccion import *
print(f'debounce: {ConfigInterseccion().debounce_emergencia_frames} frames')


In [ ]:
import os, glob, random

def procesar_interseccion(imagenes_por_camara, modelo, controlador_inter,
                           controlador_visibilidad=None, conf=0.25):
    """
    Procesa 2 imágenes (cam_sur + cam_este) y devuelve la decisión del controlador central.

    Args:
        imagenes_por_camara : dict {Camara: ruta_str o ndarray}
        modelo              : YOLO ya cargado
        controlador_inter   : instancia de ControladorInterseccion
        controlador_visibilidad: instancia opcional de DetectorVisibilidad
        conf                : umbral de confianza YOLO

    Returns:
        dict con la decisión y el detalle por cámara.
    """
    detecciones  = {}
    imgs_anotadas = {}

    for camara, img_input in imagenes_por_camara.items():
        # 1. Cargar imagen
        img = cv2.imread(img_input) if isinstance(img_input, str) else img_input.copy()

        # 2. Evaluar visibilidad
        visibilidad_ok = True
        if controlador_visibilidad is not None:
            estado_vis = controlador_visibilidad.evaluar(img)
            visibilidad_ok = estado_vis['nivel'] != 'CRITICA'

        # 3. Inferencia YOLO
        n_peatones, n_vehiculos, hay_emergencia = 0, 0, False
        img_anotada = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if visibilidad_ok:
            resultado = modelo.predict(img, conf=conf, device=DEVICE, verbose=False)[0]
            img_anotada = cv2.cvtColor(resultado.plot(), cv2.COLOR_BGR2RGB)
            if resultado.boxes is not None:
                for box in resultado.boxes:
                    cls      = int(box.cls[0])
                    conf_val = float(box.conf[0])
                    if cls == 0:
                        n_peatones += 1
                    elif cls == 1:
                        n_vehiculos += 1
                    elif cls == 2 and conf_val >= 0.5:
                        hay_emergencia = True
                        n_vehiculos   += 1
                        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                        cv2.rectangle(img_anotada, (x1, y1), (x2, y2), (255, 0, 0), 3)

        detecciones[camara] = DeteccionPorCamara(
            camara=camara,
            n_peatones=n_peatones,
            n_vehiculos=n_vehiculos,
            hay_emergencia=hay_emergencia,
            visibilidad_ok=visibilidad_ok,
        )
        imgs_anotadas[camara] = img_anotada

    # 4. Decisión del controlador central
    decision = controlador_inter.decidir(detecciones)

    # 5. Texto de resultado
    print('=' * 60)
    print(f'DECISIÓN: {decision["fase"].value}')
    print(f'Motivo:   {decision["motivo"]}')
    print('=' * 60)
    titulos_texto = {Camara.SUR: 'cam_sur (Sur→Norte)', Camara.ESTE: 'cam_este (Este→Oeste)'}
    for cam, det in detecciones.items():
        vis  = '✓' if det.visibilidad_ok else '✗ CRÍTICA'
        emer = '  🚨 EMERGENCIA' if det.hay_emergencia else ''
        print(f'{titulos_texto.get(cam, cam.value)}: {det.n_peatones} peatones, {det.n_vehiculos} vehículos [vis: {vis}]{emer}')

    # 6. Visualizar ambas cámaras lado a lado
    plt.close('all')                        # cierra figuras abiertas de celdas anteriores
    n = len(imgs_anotadas)
    fig, axes = plt.subplots(1, n, figsize=(8 * n, 6))
    if n == 1:
        axes = [axes]
    for ax, (cam, img_rgb) in zip(axes, imgs_anotadas.items()):
        det   = detecciones[cam]
        label = f'{titulos_texto.get(cam, cam.value)}\n{det.n_peatones} peat · {det.n_vehiculos} veh'
        if det.hay_emergencia:   label += '  🚨'
        if not det.visibilidad_ok: label += '  ⚠️'
        ax.imshow(img_rgb)
        ax.set_title(label, fontsize=11)
        ax.axis('off')
    fig.suptitle(f'DECISIÓN: {decision["fase"].value}  |  {decision["motivo"]}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    from IPython.display import display as ipy_display
    ipy_display(fig)
    plt.close(fig)

    return {'decision': decision, 'detecciones': detecciones}


# ─────────────────────────────────────────────────────────────
# DEMO: probar con imágenes reales de val/
# ─────────────────────────────────────────────────────────────
carpeta_val = os.path.join(DATASET_PATH, 'images', 'val')
imgs_val = (glob.glob(os.path.join(carpeta_val, '*.jpg')) +
            glob.glob(os.path.join(carpeta_val, '*.png')))

if len(imgs_val) >= 2:
    img_sur  = random.choice(imgs_val)
    img_este = random.choice([i for i in imgs_val if i != img_sur])

    controlador_inter = ControladorInterseccion()

    resultado = procesar_interseccion(
        imagenes_por_camara={Camara.SUR: img_sur, Camara.ESTE: img_este},
        modelo=modelo_eval,
        controlador_inter=controlador_inter,
        conf=0.25,
    )
else:
    print(f'⚠️  No hay suficientes imágenes en {carpeta_val}')


In [ ]:
def procesar_interseccion_video(video_sur, video_este, modelo,
                                 salida='videos/interseccion_resultado.mp4'):
    """Procesa 2 videos lado a lado con decisión del controlador de intersección."""
    cap_sur  = cv2.VideoCapture(video_sur)
    cap_este = cv2.VideoCapture(video_este)

    fps = int(cap_sur.get(cv2.CAP_PROP_FPS)) or 25
    w   = int(cap_sur.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap_sur.get(cv2.CAP_PROP_FRAME_HEIGHT))

    header_h = 80
    out_w    = w * 2
    out_h    = h + header_h

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(salida, fourcc, fps, (out_w, out_h))

    controlador = ControladorInterseccion()
    historial   = []
    frame_idx   = 0

    while True:
        ret_sur,  frame_sur  = cap_sur.read()
        ret_este, frame_este = cap_este.read()
        if not ret_sur and not ret_este:
            break
        if not ret_sur:  frame_sur  = np.zeros((h, w, 3), dtype=np.uint8)
        if not ret_este: frame_este = np.zeros((h, w, 3), dtype=np.uint8)

        frame_sur  = cv2.resize(frame_sur,  (w, h))
        frame_este = cv2.resize(frame_este, (w, h))

        r_sur  = modelo.predict(frame_sur,  conf=0.25, device=DEVICE, verbose=False)[0]
        r_este = modelo.predict(frame_este, conf=0.25, device=DEVICE, verbose=False)[0]

        def extraer(r, camara):
            n_p, n_v, emerg = 0, 0, False
            if r.boxes is not None:
                for box in r.boxes:
                    cls = int(box.cls[0])
                    if cls == 0:   n_p += 1
                    elif cls == 1: n_v += 1
                    elif cls == 2 and float(box.conf[0]) >= 0.5: emerg = True; n_v += 1
                    elif cls == 2: n_v += 1
            return DeteccionPorCamara(camara=camara, n_peatones=n_p,
                                      n_vehiculos=n_v, hay_emergencia=emerg)

        detecciones = {
            Camara.SUR:  extraer(r_sur,  Camara.SUR),
            Camara.ESTE: extraer(r_este, Camara.ESTE),
        }
        decision = controlador.decidir(detecciones)
        fase     = decision['fase']
        historial.append(fase.value)

        # Anotar frames YOLO + caja roja emergencia
        frame_sur_anot  = r_sur.plot()
        frame_este_anot = r_este.plot()
        for r, fa in [(r_sur, frame_sur_anot), (r_este, frame_este_anot)]:
            if r.boxes is not None:
                for box in r.boxes:
                    if int(box.cls[0]) == 2:
                        x1,y1,x2,y2 = map(int, box.xyxy[0].cpu().numpy())
                        cv2.rectangle(fa, (x1,y1), (x2,y2), (0,0,255), 3)

        # Borde: verde = tiene luz verde / rojo = espera / amarillo = falla
        if fase == FaseSemaforo.FALLA_SEGURA:
            c_sur = c_este = (0, 200, 255)      # amarillo ambos
        elif fase == FaseSemaforo.EMERGENCIA:
            cam_prio = decision.get('camara_prioritaria')
            c_sur  = (0,255,0) if cam_prio == Camara.SUR  else (0,0,255)
            c_este = (0,255,0) if cam_prio == Camara.ESTE else (0,0,255)
        else:
            c_sur  = (0,255,0) if fase == FaseSemaforo.FASE_SN else (0,0,255)
            c_este = (0,255,0) if fase == FaseSemaforo.FASE_EO else (0,0,255)

        borde = 8
        cv2.rectangle(frame_sur_anot,  (0,0), (w-1,h-1), c_sur,  borde)
        cv2.rectangle(frame_este_anot, (0,0), (w-1,h-1), c_este, borde)

        cv2.putText(frame_sur_anot,  'CAM SUR  (Sur->Norte)',  (10,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        cv2.putText(frame_este_anot, 'CAM ESTE (Este->Oeste)', (10,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

        # Header con decisión
        header = np.full((header_h, out_w, 3), 30, dtype=np.uint8)
        col_texto = (0,255,0) if fase in (FaseSemaforo.FASE_SN, FaseSemaforo.FASE_EO) else \
                    (0,0,255) if fase == FaseSemaforo.EMERGENCIA else (0,200,255)
        cv2.putText(header, f'DECISION: {fase.value}  |  {decision["motivo"]}',
                    (20,50), cv2.FONT_HERSHEY_SIMPLEX, 0.9, col_texto, 2)

        frame_comp = np.zeros((out_h, out_w, 3), dtype=np.uint8)
        frame_comp[:header_h, :]  = header
        frame_comp[header_h:, :w] = frame_sur_anot
        frame_comp[header_h:, w:] = frame_este_anot

        out.write(frame_comp)
        frame_idx += 1
        if frame_idx % 30 == 0:
            print(f"   Frame {frame_idx}")

    cap_sur.release()
    cap_este.release()
    out.release()
    print(f'\n✅ Video guardado: {salida}')
    print(f'   Distribución: {Counter(historial)}')

    import subprocess
    salida_h264 = salida.replace('.mp4', '_h264.mp4')
    subprocess.run(['ffmpeg','-y','-i',salida,'-vcodec','libx264','-crf','23',salida_h264],
                   capture_output=True)
    ipy_display(IPVideo(salida_h264, embed=True))


# ── Llamada ──────────────────────────────────────────────────
procesar_interseccion_video(
    video_sur='videos/Sur1_90s.mp4',
    video_este='videos/Ambu1_90s.mp4',
    modelo=modelo_eval,
    salida='videos/interseccion_resultado2.mp4'
)


---
## Sección 13 — Conclusiones y próximos pasos

### Logros
- Sistema funcional de detección de 3 clases (peatones, vehículos, vehículos de emergencia) basado en YOLOv8 con transfer learning desde COCO.
- Pipeline de **augmentation climático** con Albumentations: niebla, lluvia, blur, baja luminancia — específicamente diseñado para Concepción.
- **Detector de baja visibilidad** que mide nitidez (Laplaciano), contraste (RMS) y brillo en cada frame.
- **Falla segura**: el sistema revierte al ciclo de tiempo fijo cuando la visibilidad no es confiable, en vez de tomar decisiones arriesgadas.
- **Control de intersección con 2 cámaras**: arquitectura cam_sur (Sur→Norte) + cam_este (Este→Oeste) con `ControladorInterseccion` y 4 reglas de priorización: falla segura, emergencia, densidad asimétrica y ciclo normal.
- **Control temporal basado en frames** con bloqueo mínimo de 20s y debounce de emergencia de 10 frames consecutivos para evitar falsas activaciones.
- **Procesamiento de video** frame a frame con visualización lado a lado de ambas cámaras, bordes de color indicando prioridad de paso y re-encoding H.264 para reproducción en el notebook.
- **Reentrenamiento con 50 épocas** (exp2_yolov8s_50ep): mAP@0.5 mejoró de 0.5646 a 0.620 y mAP@0.5:0.95 de 0.4237 a 0.471.
- Pipeline end-to-end aplicable a imágenes y video.

### Limitaciones
- Dependencia de la calidad y diversidad del dataset (especialmente para vehículos con balizas en condiciones reales).
- Falsos positivos en clase `vehiculo_emergencia` reducidos con el modelo de 50 épocas, pero aún presentes — se recomienda ampliar el dataset de ambulancias y vehículos de bomberos.
- Motocicletas incluidas en el dataset (mapeadas como `vehiculo`) pero con detección mejorable; requiere mayor representación en datos de entrenamiento.
- Sin tracking entre frames: cada frame es independiente. Para producción habría que integrar ByteTrack o DeepSORT.
- Reglas de priorización son heurísticas; podrían optimizarse con aprendizaje por refuerzo (Módulo 2 — RL).
- Umbrales del detector de visibilidad fijados de forma heurística — deberían calibrarse con datos del sitio real de despliegue.
- El augmentation sintético (niebla con RandomFog) no reproduce 100 % el comportamiento óptico de la niebla real (dispersión Mie, etc.).

### Próximos pasos Despliegue
- Configurar CUDA para GPU NVIDIA disponible (TU117M) — reduciría el tiempo de inferencia de minutos a segundos por video.
- Exportar el modelo a ONNX o TensorRT para inferencia optimizada.
- Desplegar en hardware edge (Jetson Nano / Coral / Raspberry Pi 5).
- Integración con sistemas SCADA o controladores de semáforo reales.
- API REST/MQTT para comunicación con la infraestructura urbana.
- Monitoreo y reentrenamiento continuo (MLOps).
- Evaluación con cámara térmica complementaria como alternativa de respaldo en visibilidad crítica.
- Calibración de umbrales de visibilidad en terreno (datos reales de Concepción en distintas estaciones).

### Consideraciones éticas
- **Privacidad**: el sistema procesa imágenes de espacio público; debe cumplir con la Ley 19.628 sobre protección de datos personales.
- **Sesgo**: el modelo puede fallar más con personas con movilidad reducida, sillas de ruedas, bicicletas, etc. Validar la equidad antes de desplegar.
- **Falla segura**: explícitamente implementada — ante baja visibilidad o fallo del modelo, el sistema revierte al ciclo de semáforo tradicional, nunca queda bloqueado.
- **Explicabilidad**: las decisiones del controlador son transparentes y auditables (regla disparada y métricas de visibilidad visibles en cada frame).
